# Calcium Imaging — Analysis
Loads pipeline output: co-activity, proximity analysis, and an annotated video.

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = r"Z:\ephacoffice\DColameo\Ca_Anand_AllData\07_11_25_Min6WT_Binning2_2_250ms_Exp3\pipeline_output_single"

# Co-activity thresholds
CORR_THRESHOLD   = 0.4    # minimum Pearson r to consider two cells co-active
DIST_THRESHOLD   = 60     # maximum centroid distance (pixels) to consider cells "nearby"

# Video
VIDEO_FPS        = 5      # playback speed (frames/s); original data is TARGET_FPS
DFF_VMAX         = 1.0    # dF/F colormap saturation (adjust if traces are very large/small)
MAX_VIDEO_FRAMES = None   # int to cap video length (None = all frames)
SAVE_VIDEO_MP4   = False  # True = also write .mp4 (requires ffmpeg on PATH)




# Dimensionality reduction
DR_N_PCS       = 10     # PCA components to compute
DR_TIME_BIN_S  = 0      # seconds per time bin (0 = no binning, use raw frames)
DR_USE_UMAP    = True   # attempt UMAP (requires: pip install umap-learn)
# ROI activity video
ROI_VIDEO_FPS   = 15    # playback FPS (higher = faster, default raw data is TARGET_FPS)
ROI_VIDEO_ALPHA = 0.75  # ROI overlay opacity (0 = transparent, 1 = fully opaque)
ROI_VIDEO_CMAP  = 'inferno'  # colormap for dF/F values

# Burst detection
ACTIVITY_THRESHOLD = 0.3    # dF/F to consider a cell 'active' per frame
BURST_THRESHOLD    = 0.2    # fraction of cells active to call a network burst
BURST_MIN_DURATION = 0.5    # seconds — shorter bursts are filtered out
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
from pathlib import Path
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.animation as animation
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
import matplotlib.cm as cm
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from skimage.segmentation import find_boundaries
from skimage import measure
from IPython.display import HTML

plt.rcParams['figure.dpi'] = 110
plt.rcParams['animation.embed_limit'] = 512   # MB — raise if video is large

out_dir = Path(OUTPUT_DIR)

## 1 — Load pipeline outputs

In [ ]:
roi_mask  = np.load(out_dir / 'roi_mask.npy')
centroids = np.load(out_dir / 'centroids.npy')   # (n_cells, 2): [y, x]
F_raw     = np.load(out_dir / 'F_raw.npy')        # (n_cells, T)
dff       = np.load(out_dir / 'dff.npy')          # (n_cells, T)
time_axis = np.load(out_dir / 'time_axis.npy')    # (T,) seconds
mean_img  = tifffile.imread(str(out_dir / 'mean_image.tif'))
mov       = tifffile.imread(str(out_dir / 'mov_downsampled.tif')).astype(np.float32)

n_cells, T = dff.shape
H, W       = roi_mask.shape
rprops     = measure.regionprops(roi_mask)

print(f"Cells     : {n_cells}")
print(f"Frames    : {T}  ({time_axis[-1]:.0f} s)")
print(f"Frame size: {H} × {W} px")
print(f"Movie     : {mov.shape}  ({mov.nbytes/1e6:.0f} MB)")

## 2 — Overview

In [ ]:
from skimage.color import label2rgb
mean_norm = (mean_img - mean_img.min()) / (mean_img.max() - mean_img.min() + 1e-9)
overlay   = np.clip(label2rgb(roi_mask, image=mean_norm, bg_label=0, alpha=0.4), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
p1, p99 = np.percentile(mean_img, [1, 99])
axes[0].imshow(mean_img, cmap='gray', vmin=p1, vmax=p99)
axes[0].set_title('Mean image'); axes[0].axis('off')
axes[1].imshow(overlay)
axes[1].set_title(f'ROIs (n={n_cells})')
for rp in rprops:
    y, x = rp.centroid
    axes[1].text(x, y, str(rp.label), color='white', fontsize=6, ha='center', va='center')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 3 — Co-activity: pairwise correlation

In [ ]:
# Pearson correlation matrix
corr = np.corrcoef(dff)   # (n_cells, n_cells)

# Hierarchical clustering to reorder rows/cols by similarity
dist_corr = 1 - corr
np.fill_diagonal(dist_corr, 0)
dist_corr = np.clip(dist_corr, 0, None)
dist_corr = (dist_corr + dist_corr.T) / 2   # enforce exact symmetry
Z = linkage(squareform(dist_corr), method='ward')
order = dendrogram(Z, no_plot=True)['leaves']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Raw order
im = axes[0].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[0].set_title('Pairwise correlation (original order)')
axes[0].set_xlabel('Cell #'); axes[0].set_ylabel('Cell #')
plt.colorbar(im, ax=axes[0], label='Pearson r')

# Clustered order
corr_ord = corr[np.ix_(order, order)]
im2 = axes[1].imshow(corr_ord, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[1].set_title('Pairwise correlation (clustered)')
axes[1].set_xlabel('Cell # (reordered)'); axes[1].set_ylabel('Cell # (reordered)')
plt.colorbar(im2, ax=axes[1], label='Pearson r')

plt.tight_layout()
plt.show()

n_coactive = int(((corr > CORR_THRESHOLD).sum() - n_cells) / 2)
print(f"Pairs with r > {CORR_THRESHOLD}: {n_coactive}")

## 4 — Proximity vs co-activity

In [ ]:
# Pairwise Euclidean distance between centroids
dist_mat = squareform(pdist(centroids))   # (n_cells, n_cells) pixels

# Upper triangle only (no diagonal)
triu_idx = np.triu_indices(n_cells, k=1)
pairwise_dist = dist_mat[triu_idx]
pairwise_corr = corr[triu_idx]

# Classify pairs
nearby    = pairwise_dist < DIST_THRESHOLD
coactive  = pairwise_corr > CORR_THRESHOLD
both      = nearby & coactive

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scatter: distance vs correlation
colors = np.where(both, 'crimson', np.where(nearby, 'orange', np.where(coactive, 'steelblue', 'lightgray')))
axes[0].scatter(pairwise_dist, pairwise_corr, c=colors, s=12, alpha=0.6, linewidths=0)
axes[0].axhline(CORR_THRESHOLD, color='steelblue', lw=1, ls='--', label=f'r = {CORR_THRESHOLD}')
axes[0].axvline(DIST_THRESHOLD, color='orange',    lw=1, ls='--', label=f'd = {DIST_THRESHOLD} px')
axes[0].set_xlabel('Centroid distance (px)')
axes[0].set_ylabel('Pearson r')
axes[0].set_title('Proximity vs co-activity (all pairs)')
legend_els = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='crimson',    ms=7, label='nearby + co-active'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='orange',     ms=7, label='nearby only'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue',  ms=7, label='co-active only'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='lightgray',  ms=7, label='neither'),
]
axes[0].legend(handles=legend_els, fontsize=8)

# Distance matrix heatmap
im = axes[1].imshow(dist_mat, cmap='viridis_r', aspect='auto')
axes[1].set_title('Centroid distance matrix (px)')
axes[1].set_xlabel('Cell #'); axes[1].set_ylabel('Cell #')
plt.colorbar(im, ax=axes[1], label='Distance (px)')

plt.tight_layout()
plt.show()

print(f"Nearby pairs (d < {DIST_THRESHOLD} px)               : {nearby.sum()}")
print(f"Co-active pairs (r > {CORR_THRESHOLD})               : {coactive.sum()}")
print(f"Nearby AND co-active                              : {both.sum()}")

## 5 — Co-active neighbour map
Lines connect pairs that are both nearby and co-active. Line colour = correlation, width ∝ correlation.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10 * H / W))
p1, p99 = np.percentile(mean_img, [1, 99])
ax.imshow(mean_img, cmap='gray', vmin=p1, vmax=p99)

edge_cmap = cm.get_cmap('YlOrRd')
edge_norm = Normalize(vmin=CORR_THRESHOLD, vmax=1.0)

# Draw edges for nearby+co-active pairs
i_idx, j_idx = triu_idx
for i, j, d, r in zip(i_idx, j_idx, pairwise_dist, pairwise_corr):
    if d < DIST_THRESHOLD and r > CORR_THRESHOLD:
        y0, x0 = centroids[i]
        y1, x1 = centroids[j]
        ax.plot([x0, x1], [y0, y1],
                color=edge_cmap(edge_norm(r)),
                lw=1 + 3 * edge_norm(r),
                alpha=0.8, solid_capstyle='round')

# Draw cell dots
for i, rp in enumerate(rprops):
    y, x = rp.centroid
    ax.plot(x, y, 'o', color='white', ms=4, mew=0)
    ax.text(x + 3, y, str(rp.label), color='white', fontsize=6, va='center')

sm = cm.ScalarMappable(cmap=edge_cmap, norm=edge_norm)
plt.colorbar(sm, ax=ax, label='Pearson r', shrink=0.5)
ax.set_title(f'Co-active neighbours  (r > {CORR_THRESHOLD}, d < {DIST_THRESHOLD} px)')
ax.axis('off')
plt.tight_layout()
plt.show()

## 7 — Network burst detection
Identifies epochs where a large fraction of cells fire simultaneously.
Threshold: `BURST_THRESHOLD` (fraction of cells active), min duration: `BURST_MIN_DURATION` s.

In [ ]:
from scipy.ndimage import label as nd_label

# Binarize per-cell activity
active_binary = dff > ACTIVITY_THRESHOLD           # (n_cells, T)

# Population activity trace: fraction of cells active each frame
pop_activity = active_binary.mean(axis=0)          # (T,)

# ── Burst detection ───────────────────────────────────────────────────────
fps_est       = T / time_axis[-1]
min_frames    = max(1, int(BURST_MIN_DURATION * fps_est))
burst_labeled, n_raw = nd_label(pop_activity >= BURST_THRESHOLD)

bursts = []
for b in range(1, n_raw + 1):
    frames = np.where(burst_labeled == b)[0]
    if len(frames) < min_frames:
        continue
    bursts.append({
        'id'         : len(bursts) + 1,
        'frame_start': int(frames[0]),
        'frame_end'  : int(frames[-1]),
        't_start'    : float(time_axis[frames[0]]),
        't_end'      : float(time_axis[frames[-1]]),
        'duration_s' : float(time_axis[frames[-1]] - time_axis[frames[0]]),
        'peak_frac'  : float(pop_activity[frames].max()),
        'n_recruits' : int(active_binary[:, frames].any(axis=1).sum()),
    })

print(f"Activity threshold    : {ACTIVITY_THRESHOLD} dF/F per cell")
print(f"Burst threshold       : {BURST_THRESHOLD*100:.0f}% of {n_cells} cells active")
print(f"Min burst duration    : {BURST_MIN_DURATION} s  ({min_frames} frames)")
print(f"Bursts detected       : {len(bursts)}")
if bursts:
    durs = [b['duration_s'] for b in bursts]
    print(f"Duration mean ± std   : {np.mean(durs):.2f} ± {np.std(durs):.2f} s")
    if len(bursts) > 1:
        ibi = [bursts[i+1]['t_start'] - bursts[i]['t_end'] for i in range(len(bursts)-1)]
        print(f"Inter-burst interval  : {np.mean(ibi):.2f} ± {np.std(ibi):.2f} s")

# ── Plot 1: Raster + population activity trace ───────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 8),
                          gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
ax_raster, ax_pop = axes

for i in range(n_cells):
    t_on = time_axis[active_binary[i]]
    ax_raster.scatter(t_on, np.full(len(t_on), i), s=1.5, c='k', linewidths=0)
for b in bursts:
    ax_raster.axvspan(b['t_start'], b['t_end'], color='salmon', alpha=0.3, lw=0)
ax_raster.set_ylabel('Cell #')
ax_raster.set_ylim(-0.5, n_cells - 0.5)
ax_raster.invert_yaxis()
ax_raster.set_title(f'Activity raster  —  {len(bursts)} burst(s) detected')
for s in ['top', 'right']: ax_raster.spines[s].set_visible(False)

ax_pop.fill_between(time_axis, pop_activity, alpha=0.5, color='steelblue', label='pop. activity')
ax_pop.axhline(BURST_THRESHOLD, color='crimson', lw=1, ls='--',
               label=f'burst thr = {BURST_THRESHOLD}')
for b in bursts:
    ax_pop.axvspan(b['t_start'], b['t_end'], color='salmon', alpha=0.3, lw=0)
ax_pop.set_xlabel('Time (s)')
ax_pop.set_ylabel('Fraction active')
ax_pop.set_ylim(0, 1)
ax_pop.legend(fontsize=8, loc='upper right')
for s in ['top', 'right']: ax_pop.spines[s].set_visible(False)

plt.tight_layout()
plt.show()

# ── Plot 2: dF/F heatmap sorted by burst participation ────────────────────
if bursts:
    burst_participation = np.zeros(n_cells)
    for b in bursts:
        burst_participation += active_binary[:, b['frame_start']:b['frame_end']+1].mean(axis=1)
    sort_idx = np.argsort(burst_participation)[::-1]

    fig, ax = plt.subplots(figsize=(16, max(4, n_cells * 0.18)))
    im = ax.imshow(dff[sort_idx], aspect='auto', cmap='hot',
                   vmin=0, vmax=DFF_VMAX,
                   extent=[time_axis[0], time_axis[-1], n_cells, 0])
    for b in bursts:
        ax.axvspan(b['t_start'], b['t_end'], color='cyan', alpha=0.15, lw=0)
    plt.colorbar(im, ax=ax, label='dF/F')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Cell (sorted by burst participation)')
    ax.set_title('dF/F heatmap — cells sorted by burst recruitment')
    plt.tight_layout()
    plt.show()

# ── Plot 3: Burst statistics ───────────────────────────────────────────────
if len(bursts) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    durs  = [b['duration_s'] for b in bursts]
    peaks = [b['peak_frac']  for b in bursts]
    ibi   = [bursts[i+1]['t_start'] - bursts[i]['t_end'] for i in range(len(bursts)-1)]

    for ax, data, xlabel, title, color in zip(
        axes,
        [durs, peaks, ibi],
        ['Duration (s)', 'Peak fraction active', 'Inter-burst interval (s)'],
        ['Burst duration', 'Peak pop. activity', 'Inter-burst interval'],
        ['steelblue', 'coral', 'mediumseagreen'],
    ):
        ax.hist(data, bins=min(10, len(data)), color=color, edgecolor='white')
        ax.set_xlabel(xlabel); ax.set_ylabel('Count')
        ax.set_title(title)
        for s in ['top', 'right']: ax.spines[s].set_visible(False)

    plt.tight_layout()
    plt.show()
elif len(bursts) == 1:
    print("Only 1 burst — statistics plots require ≥ 2 bursts.")
else:
    print("No bursts detected. Try lowering BURST_THRESHOLD or ACTIVITY_THRESHOLD.")


## 8 — Dimensionality reduction
Two complementary views:
- **Time-resolved** — cells as features, time bins as observations → what activity *states* does the population visit?
- **Cell-resolved** — time as features, cells as observations → which cells share the same temporal programme?

Both use per-cell z-scores (normalise each cell across its own time axis) so that the signal is activity variance, not mean brightness.

In [ ]:
from sklearn.decomposition import PCA
from scipy.stats import zscore

try:
    import umap as _umap_mod
    _UMAP_AVAILABLE = True
except ImportError:
    _UMAP_AVAILABLE = False
    if DR_USE_UMAP:
        print("umap-learn not installed — run:  pip install umap-learn\nFalling back to PCA only.")

# Per-cell z-score across time — used by BOTH views
# (do NOT z-score per frame: that would scale quiet-period noise up and burst signal down)
Z = zscore(dff, axis=1)        # (n_cells, T)

# Optional time binning for the time-resolved view
fps_est = T / time_axis[-1]
if DR_TIME_BIN_S > 0:
    bf      = max(1, int(DR_TIME_BIN_S * fps_est))
    T_trim  = (T // bf) * bf
    Z_bin   = Z[:, :T_trim].reshape(n_cells, T_trim // bf, bf).mean(axis=2)
    t_bin   = time_axis[:T_trim].reshape(T_trim // bf, bf).mean(axis=1)
else:
    Z_bin, t_bin = Z, time_axis
T_bin = Z_bin.shape[1]

# Burst label per (binned) time point
burst_t = np.zeros(T_bin, dtype=bool)
for b in bursts:
    burst_t |= (t_bin >= b['t_start']) & (t_bin <= b['t_end'])

# Burst participation score per cell (recomputed here, no cross-cell dependency)
_bp = np.zeros(n_cells)
if bursts:
    for b in bursts:
        _bp += active_binary[:, b['frame_start']:b['frame_end']+1].mean(axis=1)

# ── A: Time-resolved PCA ─────────────────────────────────────────────────
n_t   = min(DR_N_PCS, n_cells - 1, T_bin - 1)
pca_t = PCA(n_components=n_t)
emb_t = pca_t.fit_transform(Z_bin.T)        # (T_bin, n_t)
ev_t  = pca_t.explained_variance_ratio_ * 100

# ── B: Cell-resolved PCA ─────────────────────────────────────────────────
n_c   = min(DR_N_PCS, n_cells - 1, T - 1)
pca_c = PCA(n_components=n_c)
emb_c = pca_c.fit_transform(Z)              # (n_cells, n_c)
ev_c  = pca_c.explained_variance_ratio_ * 100

print(f"Time-resolved PCA  — PC1 explains {ev_t[0]:.1f}%  (top {n_t} PCs: {ev_t.sum():.1f}%)")
print(f"Cell-resolved PCA  — PC1 explains {ev_c[0]:.1f}%  (top {n_c} PCs: {ev_c.sum():.1f}%)")

# ── Plots ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 11))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.38)

# Row 0 — Time-resolved
# Scree
ax = fig.add_subplot(gs[0, 0])
ax.bar(range(1, len(ev_t)+1), ev_t, color='steelblue', edgecolor='white')
ax.plot(range(1, len(ev_t)+1), np.cumsum(ev_t), 'k-o', ms=4, label='cumulative')
ax.set_xlabel('PC'); ax.set_ylabel('Variance (%)')
ax.set_title('Scree — time-resolved')
ax.legend(fontsize=7)
for s in ['top','right']: ax.spines[s].set_visible(False)

# PC1/2 scatter coloured by time, bursts highlighted
ax = fig.add_subplot(gs[0, 1])
sc = ax.scatter(emb_t[~burst_t, 0], emb_t[~burst_t, 1],
                c=t_bin[~burst_t], cmap='viridis', s=8, alpha=0.6, linewidths=0)
if burst_t.any():
    ax.scatter(emb_t[burst_t, 0], emb_t[burst_t, 1],
               c='crimson', s=20, alpha=0.9, linewidths=0, label='burst', zorder=3)
    ax.legend(fontsize=7)
plt.colorbar(sc, ax=ax, label='Time (s)', shrink=0.8)
ax.set_xlabel(f'PC1 ({ev_t[0]:.1f}%)'); ax.set_ylabel(f'PC2 ({ev_t[1]:.1f}%)')
ax.set_title('Time-resolved embedding')
for s in ['top','right']: ax.spines[s].set_visible(False)

# PC1-3 traces over time
ax = fig.add_subplot(gs[0, 2:])
colors3 = ['steelblue', 'coral', 'mediumseagreen']
n_show  = min(3, n_t)
scale   = emb_t[:, 0].std() * 3
for k in range(n_show):
    offset = k * scale
    ax.plot(t_bin, emb_t[:, k] - offset, color=colors3[k], lw=0.9, label=f'PC{k+1}')
for b in bursts:
    ax.axvspan(b['t_start'], b['t_end'], color='salmon', alpha=0.3, lw=0)
ax.set_xlabel('Time (s)'); ax.set_ylabel('PC score (offset)')
ax.set_title('Top PCs over time  (burst epochs = salmon)')
ax.legend(fontsize=8)
for s in ['top','right']: ax.spines[s].set_visible(False)

# Row 1 — Cell-resolved
# Scree
ax = fig.add_subplot(gs[1, 0])
ax.bar(range(1, len(ev_c)+1), ev_c, color='darkorange', edgecolor='white')
ax.plot(range(1, len(ev_c)+1), np.cumsum(ev_c), 'k-o', ms=4, label='cumulative')
ax.set_xlabel('PC'); ax.set_ylabel('Variance (%)')
ax.set_title('Scree — cell-resolved')
ax.legend(fontsize=7)
for s in ['top','right']: ax.spines[s].set_visible(False)

# Cell PC1/2 scatter coloured by burst participation
ax = fig.add_subplot(gs[1, 1])
sc2 = ax.scatter(emb_c[:, 0], emb_c[:, 1], c=_bp, cmap='YlOrRd',
                 s=50, edgecolors='k', linewidths=0.4, zorder=3)
for i, rp in enumerate(rprops):
    ax.annotate(str(rp.label), (emb_c[i, 0], emb_c[i, 1]),
                fontsize=6, ha='center', va='bottom', color='gray')
plt.colorbar(sc2, ax=ax, label='Burst participation', shrink=0.8)
ax.set_xlabel(f'PC1 ({ev_c[0]:.1f}%)'); ax.set_ylabel(f'PC2 ({ev_c[1]:.1f}%)')
ax.set_title('Cell-resolved embedding')
for s in ['top','right']: ax.spines[s].set_visible(False)

# Spatial loading maps for PC1 and PC2
p1_v, p99_v = np.percentile(mean_img, [1, 99])
load_cmap   = cm.get_cmap('coolwarm')

for col_idx, (pc_idx, title) in enumerate([(0, 'PC1 loading'), (1, 'PC2 loading')]):
    ax = fig.add_subplot(gs[1, 2 + col_idx])
    ax.imshow(mean_img, cmap='gray', vmin=p1_v, vmax=p99_v)
    loading   = emb_c[:, pc_idx]
    load_norm = plt.Normalize(vmin=-np.abs(loading).max(), vmax=np.abs(loading).max())
    for i, rp in enumerate(rprops):
        y, x = rp.centroid
        ax.plot(x, y, 'o', color=load_cmap(load_norm(loading[i])), ms=8,
                mew=0.6, mec='k')
        ax.text(x + 4, y, str(rp.label), color='white', fontsize=5, va='center')
    sm = cm.ScalarMappable(cmap=load_cmap, norm=load_norm)
    plt.colorbar(sm, ax=ax, label=title, shrink=0.8)
    ax.set_title(f'Spatial map — {title}')
    ax.axis('off')

plt.suptitle('Dimensionality reduction (PCA)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# ── UMAP (optional) ───────────────────────────────────────────────────────
if DR_USE_UMAP and _UMAP_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Time-resolved UMAP
    nn_t  = min(15, T_bin - 1)
    emb_umap_t = _umap_mod.UMAP(n_components=2, n_neighbors=nn_t,
                                  random_state=42).fit_transform(Z_bin.T)
    sc = axes[0].scatter(emb_umap_t[~burst_t, 0], emb_umap_t[~burst_t, 1],
                         c=t_bin[~burst_t], cmap='viridis', s=8, alpha=0.6, linewidths=0)
    if burst_t.any():
        axes[0].scatter(emb_umap_t[burst_t, 0], emb_umap_t[burst_t, 1],
                        c='crimson', s=20, alpha=0.9, linewidths=0, label='burst', zorder=3)
        axes[0].legend(fontsize=8)
    plt.colorbar(sc, ax=axes[0], label='Time (s)')
    axes[0].set_xlabel('UMAP 1'); axes[0].set_ylabel('UMAP 2')
    axes[0].set_title('Time-resolved UMAP (cells = features)')
    for s in ['top','right']: axes[0].spines[s].set_visible(False)

    # Cell-resolved UMAP (skip if too few cells)
    if n_cells >= 10:
        nn_c  = min(15, n_cells - 1)
        emb_umap_c = _umap_mod.UMAP(n_components=2, n_neighbors=nn_c,
                                      random_state=42).fit_transform(Z)
        sc2 = axes[1].scatter(emb_umap_c[:, 0], emb_umap_c[:, 1], c=_bp, cmap='YlOrRd',
                              s=50, edgecolors='k', linewidths=0.4)
        for i, rp in enumerate(rprops):
            axes[1].annotate(str(rp.label), (emb_umap_c[i, 0], emb_umap_c[i, 1]),
                             fontsize=6, ha='center', va='bottom', color='gray')
        plt.colorbar(sc2, ax=axes[1], label='Burst participation')
        axes[1].set_xlabel('UMAP 1'); axes[1].set_ylabel('UMAP 2')
        axes[1].set_title('Cell-resolved UMAP (time = features)')
        for s in ['top','right']: axes[1].spines[s].set_visible(False)
    else:
        axes[1].text(0.5, 0.5, f'Cell-resolved UMAP skipped\n(n_cells={n_cells} < 10)',
                     ha='center', va='center', transform=axes[1].transAxes)
        axes[1].axis('off')

    plt.suptitle('Dimensionality reduction (UMAP)', fontsize=12)
    plt.tight_layout()
    plt.show()


## 9 — ROI activity video
Renders an MP4 using only the saved pipeline outputs — no raw TIFF needed.
Each ROI mask is filled with a colour proportional to its dF/F, overlaid on the mean image.
A population-activity trace runs below with a moving cursor.

In [ ]:
import subprocess
import matplotlib.animation as _anim
from IPython.display import Video as _IVideo

# ── Codec detection (NVENC → libx264 fallback) ───────────────────────────
def _probe_nvenc():
    try:
        r = subprocess.run(
            ['ffmpeg', '-f', 'lavfi', '-i', 'nullsrc=s=64x64:r=1',
             '-vframes', '1', '-c:v', 'h264_nvenc', '-f', 'null', '-'],
            capture_output=True, timeout=10)
        return r.returncode == 0
    except Exception:
        return False

_nvenc  = _probe_nvenc()
_codec  = 'h264_nvenc' if _nvenc else 'libx264'
_xargs  = ['-preset', 'fast', '-b:v', '8M'] if _nvenc else ['-preset', 'fast', '-crf', '18']
print(f"Encoder: {_codec}")

# ── Pre-compute rendering helpers (done once, outside the frame loop) ─────
_p1, _p99 = np.percentile(mean_img, [1, 99])
_bg   = np.clip((mean_img - _p1) / (_p99 - _p1 + 1e-9), 0, 1).astype(np.float32)
_bg3  = np.stack([_bg]*3, axis=-1)                    # (H, W, 3) float32 grayscale

_cmap       = cm.get_cmap(ROI_VIDEO_CMAP)
_cell_map   = (roi_mask - 1).astype(np.int32)         # 0-indexed; -1 = background
_roi_ys, _roi_xs = np.where(roi_mask > 0)
_ci         = _cell_map[_roi_ys, _roi_xs]             # cell index per ROI pixel

_dff_norm   = np.clip(dff, 0, DFF_VMAX) / DFF_VMAX   # (n_cells, T), pre-normalised

T_render = T if MAX_VIDEO_FRAMES is None else min(T, MAX_VIDEO_FRAMES)

def _render(t):
    frame  = _bg3.copy()
    vals   = _dff_norm[_ci, t]                         # (n_roi_pixels,) in [0, 1]
    colors = _cmap(vals)[:, :3].astype(np.float32)
    frame[_roi_ys, _roi_xs] = (ROI_VIDEO_ALPHA * colors
                                + (1 - ROI_VIDEO_ALPHA) * frame[_roi_ys, _roi_xs])
    return (np.clip(frame, 0, 1) * 255).astype(np.uint8)

# Population activity (recompute if burst section was skipped)
try:
    _pop    = pop_activity[:T_render]
    _bursts = bursts
except NameError:
    _act    = dff > ACTIVITY_THRESHOLD
    _pop    = _act.mean(axis=0)[:T_render]
    _bursts = []

# ── Build figure ──────────────────────────────────────────────────────────
fig_v = plt.figure(figsize=(9, 10), facecolor='k')
gs_v  = gridspec.GridSpec(2, 1, figure=fig_v,
                           height_ratios=[6, 1], hspace=0.04,
                           left=0.02, right=0.91, top=0.97, bottom=0.06)

ax_im  = fig_v.add_subplot(gs_v[0])
ax_tr  = fig_v.add_subplot(gs_v[1])

# Image panel
_im_h   = ax_im.imshow(_render(0), aspect='equal', interpolation='nearest')
ax_im.axis('off')
_sm     = cm.ScalarMappable(cmap=_cmap, norm=plt.Normalize(vmin=0, vmax=DFF_VMAX))
plt.colorbar(_sm, ax=ax_im, label='dF/F', shrink=0.35, pad=0.01, fraction=0.025)
_ttl    = ax_im.set_title(f't = {time_axis[0]:.1f} s', color='white', fontsize=9, pad=3)

# Population trace panel
ax_tr.set_facecolor('k')
ax_tr.fill_between(time_axis[:T_render], _pop, alpha=0.55, color='steelblue')
for b in _bursts:
    ax_tr.axvspan(b['t_start'], b['t_end'], color='salmon', alpha=0.4, lw=0)
_cursor, = ax_tr.plot([time_axis[0]]*2, [0, _pop.max()*1.15 or 0.05],
                       color='crimson', lw=1.5, zorder=5)
ax_tr.set_xlim(time_axis[0], time_axis[T_render - 1])
ax_tr.set_ylim(0, max(_pop.max() * 1.2, 0.05))
ax_tr.set_xlabel('Time (s)', color='white', fontsize=8)
ax_tr.set_ylabel('Frac. active', color='white', fontsize=7)
ax_tr.tick_params(colors='white', labelsize=7)
for sp in ax_tr.spines.values():
    sp.set_edgecolor('gray')

# ── Render & save ─────────────────────────────────────────────────────────
out_mp4 = out_dir / 'roi_activity_video.mp4'
writer  = _anim.FFMpegWriter(fps=ROI_VIDEO_FPS, codec=_codec, extra_args=_xargs)

print(f"Rendering {T_render} frames @ {ROI_VIDEO_FPS} fps …")
with writer.saving(fig_v, str(out_mp4), dpi=110):
    for t in range(T_render):
        _im_h.set_data(_render(t))
        _cursor.set_xdata([time_axis[t], time_axis[t]])
        _ttl.set_text(f't = {time_axis[t]:.1f} s')
        writer.grab_frame()
        if t % 100 == 0:
            print(f"  {t}/{T_render}", end='\r')

plt.close(fig_v)
print(f"\nSaved → {out_mp4}  ({out_mp4.stat().st_size/1e6:.1f} MB)")

_IVideo(str(out_mp4), embed=True, width=750)


## 10 — Annotated video (raw pipeline)
Play the MP4 generated by `ca_pipeline.ipynb` (Step 11). No raw data reload needed.

In [ ]:
from IPython.display import Video

mp4_path = out_dir / 'ca_video.mp4'
assert mp4_path.exists(), f"Video not found: {mp4_path}\nRun Step 11 in ca_pipeline.ipynb first."
print(f"Playing: {mp4_path}  ({mp4_path.stat().st_size/1e6:.1f} MB)")
Video(str(mp4_path), embed=True, width=900)